# Phase 1: Basic Scalar Kalman Filter

This notebook demonstrates the fundamental scalar Kalman filter applied to prediction market data.
We filter noisy market prices to estimate the true underlying probability of events.

## Outline
1. Load market data (live API or synthetic fallback)
2. Run the scalar Kalman filter with various Q/R parameters
3. Visualize: filtered vs raw, Kalman gain, innovations, parameter sensitivity, SNR
4. MLE parameter estimation with likelihood surface

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data.synthetic import generate_random_walk, generate_step_change
from src.data.market_fetcher import MarketFetcher
from src.filters.scalar_kalman import ScalarKalmanFilter
from src.analysis.visualization import (
    plot_filtered_vs_raw, plot_kalman_gain, plot_innovations,
    plot_parameter_sensitivity, plot_snr_improvement, plot_likelihood_surface,
)

sns.set_theme(style='whitegrid')
%matplotlib inline

## 1. Load Market Data

We attempt to fetch live data from Polymarket. If the API is unavailable,
we fall back to synthetic data with known ground truth.

In [ ]:
# Try live data first, fall back to synthetic
use_synthetic = False
try:
    fetcher = MarketFetcher()
    markets = fetcher.find_liquid_markets(n=5, min_volume=50000)
    if markets:
        market = markets[0]
        history = fetcher.fetch_price_history(market.market_id, market.question)
        if len(history.points) > 50:
            observations = np.array([p.price for p in history.points])
            timestamps = np.array([p.timestamp for p in history.points])
            market_title = market.question
            print(f'Loaded {len(observations)} points for: {market_title}')
        else:
            use_synthetic = True
    else:
        use_synthetic = True
except Exception as e:
    print(f'API unavailable ({e}), using synthetic data')
    use_synthetic = True

if use_synthetic:
    print('Using synthetic data with known ground truth')
    synth = generate_random_walk(n_steps=500, Q=1e-4, R=2e-3, x0=0.55, seed=42)
    observations = synth.observations
    timestamps = None
    market_title = 'Synthetic Market (Q=1e-4, R=2e-3)'
    true_states = synth.true_states

print(f'Data shape: {observations.shape}')
print(f'Price range: [{observations.min():.3f}, {observations.max():.3f}]')

## 2. Run the Scalar Kalman Filter

In [ ]:
# Run with default parameters
Q, R = 1e-4, 1e-3
kf = ScalarKalmanFilter(Q=Q, R=R)
result = kf.filter(observations, timestamps=timestamps)

print(f'Filter parameters: Q={Q:.0e}, R={R:.0e}')
print(f'Final state: x={result.states[-1]:.4f}')
print(f'Final covariance: P={result.covariances[-1]:.2e}')
print(f'Steady-state gain: K={result.gains[-1]:.4f}')

## 3. Plot 1: Filtered vs Raw Price

In [ ]:
fig = plot_filtered_vs_raw(result, title=f'Kalman Filtered Probability: {market_title}')

# If synthetic data, also plot the true state
if use_synthetic:
    ax = fig.axes[0]
    ax.plot(true_states, color='green', linewidth=1.0, alpha=0.8, label='True state')
    ax.legend()

plt.show()

## 4. Plot 2: Kalman Gain Over Time

In [ ]:
fig = plot_kalman_gain(result, title='Kalman Gain Convergence')
plt.show()

## 5. Plot 3: Innovation Sequence (Filter Diagnostics)

In [ ]:
fig = plot_innovations(result)
plt.show()

## 6. Plot 4: Parameter Sensitivity

In [ ]:
fig = plot_parameter_sensitivity(observations)
plt.show()

## 7. Plot 5: Signal-to-Noise Improvement

In [ ]:
fig = plot_snr_improvement(result)
plt.show()

## 8. MLE Parameter Estimation

Estimate optimal Q and R from data using maximum likelihood on the innovation sequence.

In [ ]:
from src.filters.parameter_estimation import estimate_parameters, likelihood_surface

Q_hat, R_hat = estimate_parameters(observations)
print(f'MLE estimates: Q={Q_hat:.2e}, R={R_hat:.2e}')

if use_synthetic:
    print(f'True values:   Q={synth.Q_true:.2e}, R={synth.R_true:.2e}')
    print(f'Q error: {abs(Q_hat - synth.Q_true) / synth.Q_true:.1%}')
    print(f'R error: {abs(R_hat - synth.R_true) / synth.R_true:.1%}')

In [ ]:
# Log-likelihood surface
Q_range = np.logspace(-7, -2, 30)
R_range = np.logspace(-5, -1, 30)
ll_grid = likelihood_surface(observations, Q_range, R_range)

fig = plot_likelihood_surface(Q_range, R_range, ll_grid, Q_hat, R_hat)
plt.show()

## 9. Compare MLE vs Hand-Tuned Parameters

In [ ]:
# Run filter with MLE parameters
kf_mle = ScalarKalmanFilter(Q=Q_hat, R=R_hat)
result_mle = kf_mle.filter(observations)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8))

# Top: hand-tuned
plot_filtered_vs_raw(result, title=f'Hand-tuned: Q={Q:.0e}, R={R:.0e}', ax=ax1)

# Bottom: MLE
plot_filtered_vs_raw(result_mle, title=f'MLE: Q={Q_hat:.2e}, R={R_hat:.2e}', ax=ax2)

plt.tight_layout()
plt.show()